### GCRL implementation

**Notebook notes (updated for the continuous Four Rooms task):**
- the copied MountainCar environment has been replaced with a continuous Four Rooms environment
- `obs_dim` now means **state dimension only** (`[row, column]`), while the goal is passed separately into the actor and critic
- the fixed goal `s*` is taken directly from the environment, so the actor/critic always see the right goal shape
- the diagnostics and rollout cells now plot Four Rooms coordinates instead of MountainCar position/velocity
- the core GCRL logic is unchanged: contrastive critic on future goals + goal-conditioned actor + entropy tuning

In [ ]:
import numpy as np
import torch
import gymnasium as gym
from dataclasses import dataclass


FOUR_ROOMS_WALLS = np.array([
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0],
], dtype=np.int8)

STATE_LABELS = ("row", "column")
MIN_CELL_OFFSET = 0.05
MAX_CELL_OFFSET = 0.95
MIN_START_DISTANCE_MULTIPLIER = 2.0


def resize_walls(walls, factor):
    row_indices = np.repeat(np.arange(walls.shape[0]), factor)
    col_indices = np.repeat(np.arange(walls.shape[1]), factor)
    return walls[row_indices][:, col_indices]


class ContinuousFourRoomsEnv(gym.Env):
    """Continuous Four Rooms with sparse single-goal reward."""

    metadata = {"render_modes": []}

    def __init__(
        self,
        resize_factor=4,
        max_episode_steps=400,
        action_noise_std=0.01,
        goal_tolerance=0.5,
        fixed_goal_cell=(1, 9),
        fixed_start_cell=None,
    ):
        super().__init__()
        self.resize_factor = int(resize_factor)
        self._walls = resize_walls(FOUR_ROOMS_WALLS, self.resize_factor)
        self.height, self.width = self._walls.shape
        self.max_episode_steps = int(max_episode_steps)
        self.action_noise_std = float(action_noise_std)
        self.goal_tolerance = float(goal_tolerance)
        self.fixed_goal_cell = tuple(fixed_goal_cell)
        self.fixed_start_cell = None if fixed_start_cell is None else tuple(fixed_start_cell)
        self.num_substeps = 10

        self.action_space = gym.spaces.Box(
            low=np.array([-1.0, -1.0], dtype=np.float32),
            high=np.array([1.0, 1.0], dtype=np.float32),
            dtype=np.float32,
        )
        self.observation_space = gym.spaces.Box(
            low=np.zeros(2, dtype=np.float32),
            high=np.array([self.height, self.width], dtype=np.float32),
            dtype=np.float32,
        )

        self.goal = self._cell_center(self.fixed_goal_cell)
        self.state = self.goal.copy()
        self.elapsed_steps = 0

    @property
    def walls(self):
        return self._walls

    def _cell_center(self, cell):
        row, col = cell
        return np.array([
            (row + 0.5) * self.resize_factor,
            (col + 0.5) * self.resize_factor,
        ], dtype=np.float32)

    def _sample_empty_state(self):
        free_rows, free_cols = np.where(self._walls == 0)
        idx = self.np_random.integers(len(free_rows))
        state = np.array([free_rows[idx], free_cols[idx]], dtype=np.float32)
        state += self.np_random.uniform(MIN_CELL_OFFSET, MAX_CELL_OFFSET, size=2).astype(np.float32)
        return state

    def _is_blocked(self, state):
        state = np.asarray(state, dtype=np.float32)
        if np.any(state < self.observation_space.low) or np.any(state >= self.observation_space.high):
            return True
        row, col = np.floor(state).astype(int)
        return bool(self._walls[row, col] == 1)

    def _get_obs(self):
        return self.state.astype(np.float32).copy()

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.elapsed_steps = 0
        self.goal = self._cell_center(self.fixed_goal_cell)

        if self.fixed_start_cell is not None:
            self.state = self._cell_center(self.fixed_start_cell)
        else:
            start_retry_limit = 1000
            self.state = self._sample_empty_state()
            retries = 0
            while np.linalg.norm(self.state - self.goal) <= MIN_START_DISTANCE_MULTIPLIER * self.goal_tolerance:
                self.state = self._sample_empty_state()
                retries += 1
                if retries >= start_retry_limit:
                    raise RuntimeError("Could not sample a start state far enough from the fixed goal.")

        info = {
            "goal": self.goal.copy(),
            "distance_to_goal": float(np.linalg.norm(self.state - self.goal)),
        }
        return self._get_obs(), info

    def step(self, action):
        action = np.asarray(action, dtype=np.float32).reshape(2)
        action = np.clip(action, self.action_space.low, self.action_space.high)

        if self.action_noise_std > 0:
            action = action + self.np_random.normal(0.0, self.action_noise_std, size=2).astype(np.float32)
            action = np.clip(action, self.action_space.low, self.action_space.high)

        dt = 1.0 / self.num_substeps
        for _ in range(self.num_substeps):
            candidate = self.state.copy()
            candidate[0] += dt * action[0]
            candidate[1] += dt * action[1]
            if not self._is_blocked(candidate):
                self.state = candidate

        self.elapsed_steps += 1
        distance_to_goal = float(np.linalg.norm(self.state - self.goal))
        terminated = distance_to_goal <= self.goal_tolerance
        truncated = self.elapsed_steps >= self.max_episode_steps
        reward = 1.0 if terminated else 0.0

        info = {
            "goal": self.goal.copy(),
            "distance_to_goal": distance_to_goal,
        }
        return self._get_obs(), reward, terminated, truncated, info


@dataclass
class ReplayBatch:
    obs: torch.Tensor
    actions: torch.Tensor
    rewards: torch.Tensor
    next_obs: torch.Tensor
    terminated: torch.Tensor
    truncated: torch.Tensor
    episode_id: torch.Tensor
    timestep: torch.Tensor
    indices: torch.Tensor

    def __len__(self):
        return self.obs.shape[0]


class TrajectoryReplayBuffer:
    def __init__(self, capacity, obs_dim, action_dim, device="cpu"):
        self.capacity = capacity
        self.obs_dim = obs_dim
        self.action_dim = action_dim
        self.device = device

        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions = np.zeros((capacity, action_dim), dtype=np.float32)
        self.rewards = np.zeros((capacity, 1), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.terminated = np.zeros((capacity, 1), dtype=np.float32)
        self.truncated = np.zeros((capacity, 1), dtype=np.float32)

        self.episode_id = np.full((capacity,), -1, dtype=np.int64)
        self.timestep = np.full((capacity,), -1, dtype=np.int64)

        self.pos = 0
        self.size = 0
        self.full = False

        self.current_episode_id = 0
        self.episode_to_indices = {}

    def __len__(self):
        return self.size

    def add_episode(self, episode):
        ep_id = self.current_episode_id
        self.current_episode_id += 1

        ep_indices = []

        T = len(episode["obs"])
        for t in range(T):
            idx = self.pos

            if self.full:
                old_ep = self.episode_id[idx]
                if old_ep in self.episode_to_indices:
                    try:
                        self.episode_to_indices[old_ep].remove(idx)
                        if len(self.episode_to_indices[old_ep]) == 0:
                            del self.episode_to_indices[old_ep]
                    except ValueError:
                        pass

            self.obs[idx] = np.asarray(episode["obs"][t], dtype=np.float32)
            self.actions[idx] = np.asarray(episode["actions"][t], dtype=np.float32).reshape(-1)
            self.rewards[idx] = np.asarray([episode["rewards"][t]], dtype=np.float32)
            self.next_obs[idx] = np.asarray(episode["next_obs"][t], dtype=np.float32)
            self.terminated[idx] = np.asarray([episode["terminated"][t]], dtype=np.float32)
            self.truncated[idx] = np.asarray([episode["truncated"][t]], dtype=np.float32)

            self.episode_id[idx] = ep_id
            self.timestep[idx] = t

            ep_indices.append(idx)

            self.pos = (self.pos + 1) % self.capacity
            if self.size < self.capacity:
                self.size += 1
            else:
                self.full = True

        self.episode_to_indices[ep_id] = ep_indices

    def sample(self, batch_size):
        assert self.size > 0, "Buffer is empty"
        idxs = np.random.randint(0, self.size, size=batch_size)

        return ReplayBatch(
            obs=torch.tensor(self.obs[idxs], device=self.device),
            actions=torch.tensor(self.actions[idxs], device=self.device),
            rewards=torch.tensor(self.rewards[idxs], device=self.device),
            next_obs=torch.tensor(self.next_obs[idxs], device=self.device),
            terminated=torch.tensor(self.terminated[idxs], device=self.device),
            truncated=torch.tensor(self.truncated[idxs], device=self.device),
            episode_id=torch.tensor(self.episode_id[idxs], device=self.device),
            timestep=torch.tensor(self.timestep[idxs], device=self.device),
            indices=torch.tensor(idxs, device=self.device),
        )

    def sample_future_goal_batch(self, batch_size, min_k=1, max_k=None, gamma=0.99):
        assert self.size > 0, "Buffer is empty"

        valid_indices = []
        future_goal_indices = []

        tries = 0
        max_tries = batch_size * 20

        while len(valid_indices) < batch_size and tries < max_tries:
            idx = np.random.randint(0, self.size)
            ep_id = self.episode_id[idx]
            t = self.timestep[idx]

            if ep_id == -1 or ep_id not in self.episode_to_indices:
                tries += 1
                continue

            ep_idxs = self.episode_to_indices[ep_id]
            ep_len = len(ep_idxs)

            if t >= ep_len - 1:
                tries += 1
                continue

            max_valid_k = ep_len - 1 - t
            if max_k is not None:
                max_valid_k = min(max_valid_k, max_k)

            if max_valid_k < min_k:
                tries += 1
                continue

            geom_k = int(np.random.geometric(p=1.0 - gamma))
            k = max(min_k, min(geom_k, max_valid_k))
            future_t = t + k
            future_idx = ep_idxs[future_t]

            valid_indices.append(idx)
            future_goal_indices.append(future_idx)
            tries += 1

        assert len(valid_indices) > 0, "Could not sample valid future-goal pairs"

        idxs = np.array(valid_indices, dtype=np.int64)
        g_idxs = np.array(future_goal_indices, dtype=np.int64)

        batch = {
            "obs": torch.tensor(self.obs[idxs], device=self.device),
            "actions": torch.tensor(self.actions[idxs], device=self.device),
            "next_obs": torch.tensor(self.next_obs[idxs], device=self.device),
            "goals": torch.tensor(self.obs[g_idxs], device=self.device),
            "rewards": torch.tensor(self.rewards[idxs], device=self.device),
            "terminated": torch.tensor(self.terminated[idxs], device=self.device),
            "truncated": torch.tensor(self.truncated[idxs], device=self.device),
            "episode_id": torch.tensor(self.episode_id[idxs], device=self.device),
            "timestep": torch.tensor(self.timestep[idxs], device=self.device),
            "future_timestep": torch.tensor(self.timestep[g_idxs], device=self.device),
            "indices": torch.tensor(idxs, device=self.device),
            "goal_indices": torch.tensor(g_idxs, device=self.device),
        }
        return batch

    def sample_negative_future_goals(self, batch_size):
        idxs = np.random.randint(0, self.size, size=batch_size)
        return torch.tensor(self.obs[idxs], device=self.device)

    def sample_positive_future_goal(self, episode_index, timestep, k, gamma=0.99):
        if episode_index not in self.episode_to_indices:
            raise ValueError(f"Episode index {episode_index} not found in buffer")

        ep_idxs = self.episode_to_indices[episode_index]
        ep_len = len(ep_idxs)

        if timestep >= ep_len - 1:
            raise ValueError(f"Timestep {timestep} is out of bounds for episode of length {ep_len}")

        max_valid_k = ep_len - 1 - timestep
        if k > max_valid_k:
            raise ValueError(f"k={k} is too large for episode of length {ep_len} at timestep {timestep}")

        geometric = torch.distributions.Geometric(probs=torch.tensor(1 - gamma))
        steps_ahead = int(geometric.sample().item()) + 1
        steps_ahead = min(steps_ahead, max_valid_k)
        print(f"Sampled k={steps_ahead} from geometric distribution with gamma={gamma}")
        future_t = timestep + steps_ahead
        future_idx = ep_idxs[future_t]
        print(self.obs[future_idx])
        return torch.tensor(self.obs[future_idx], device=self.device)

    def stats(self):
        return {
            "size": self.size,
            "capacity": self.capacity,
            "num_episodes": len(self.episode_to_indices),
            "current_episode_id": self.current_episode_id,
        }

### 2. Generate Four Rooms replay data

In [ ]:
EPISODES = 100
MAX_HORIZON = 400
BUFFER_CAPACITY = 500000
RESIZE_FACTOR = 4
ACTION_NOISE_STD = 0.01
GOAL_TOLERANCE = 0.5
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

env = ContinuousFourRoomsEnv(
    resize_factor=RESIZE_FACTOR,
    max_episode_steps=MAX_HORIZON,
    action_noise_std=ACTION_NOISE_STD,
    goal_tolerance=GOAL_TOLERANCE,
)

obs_dim = env.observation_space.shape[0]   # state only: [row, column]
action_dim = env.action_space.shape[0]     # continuous 2-D action

replay_buffer = TrajectoryReplayBuffer(
    capacity=BUFFER_CAPACITY,
    obs_dim=obs_dim,
    action_dim=action_dim,
    device=DEVICE,
)
print(DEVICE)
print(f"Four Rooms size: {env.height} x {env.width} | fixed goal s*: {env.goal}")
episodes = []

for epi in range(EPISODES):
    obs, _ = env.reset()
    done = False

    ep = {k: [] for k in ["obs", "actions", "rewards", "next_obs", "terminated", "truncated"]}

    while not done:
        action = env.action_space.sample()
        next_obs, reward, term, trunc, info = env.step(action)

        ep["obs"].append(obs.astype(np.float32))
        ep["actions"].append(action.astype(np.float32))
        ep["rewards"].append(np.float32(reward))
        ep["next_obs"].append(next_obs.astype(np.float32))
        ep["terminated"].append(np.float32(term))
        ep["truncated"].append(np.float32(trunc))

        obs = next_obs
        done = term or trunc

    ep_np = {k: np.asarray(v, dtype=np.float32) for k, v in ep.items()}
    episodes.append(ep_np)
    replay_buffer.add_episode(ep_np)

env.close()

print("Collected episodes:", len(episodes))
print("Replay stats:", replay_buffer.stats())

### 3. Check future-goal sampling from the replay buffer

In [ ]:
batch = replay_buffer.sample(batch_size=256)
s = batch.obs
a = batch.actions

episode_idx = batch.episode_id
timestep = batch.timestep
s_pos_next = replay_buffer.sample_positive_future_goal(episode_idx[0].item(), timestep[0].item(), k=10)
s_neg_next = replay_buffer.sample_negative_future_goals(batch_size=256)

positive_dot_product = torch.dot(s[0], s_pos_next)
print("positive dot product (raw):", positive_dot_product.item())

for s_item, a_item, s_next_item in zip(batch.obs[:10], batch.actions[:10], batch.next_obs[:10]):
    print("s:", s_item.cpu().numpy(), "a:", a_item.cpu().numpy(), "s_next:", s_next_item.cpu().numpy())

### 4. The implementation of A Single Goal is All you Need.

###  $\phi(s,a)$

In [ ]:
# φ(s, a) is the state-action encoder used by the contrastive critic.
# It does NOT predict the next state directly; it maps (s, a) into a latent space
# where future goals should score highly and unrelated goals should score poorly.

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class StateActionRepresentationModel(nn.Module):

    def __init__(self, obs_dim, action_dim, hidden_dim=256, output_dim=64, hidden_layers=5, normalise=False):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim + action_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(hidden_layers - 2)])
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.normalise = normalise

    def forward(self, x):
        x = F.relu(self.fc1(x))
        for layer in self.hidden_layers:
            x = F.relu(layer(x))
        x = self.fc_out(x)
        if self.normalise:
            x = F.normalize(x, dim=-1)
        return x

 ### $\psi(g)$

In [ ]:
class GoalRepresentationModel(nn.Module): # symmetric counterpart to StateActionRepresentationModel — maps a future/goal state sf to an embedding ψ(sf)

    def __init__(self, obs_dim, hidden_dim=256, hidden_layers=5, output_dim=64, normalise=False): # same architecture choices as phi to keep the embedding space consistent
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(hidden_layers - 2)])
        self.fc_out = nn.Linear(hidden_dim, output_dim) # output in the same embedding space as StateActionRepresentationModel so that the dot product phi(s,a)^T psi(sf) is well-defined
        self.normalise = normalise

    def forward(self, x):
        x = F.relu(self.fc1(x))
        for layer in self.hidden_layers:
            x = F.relu(layer(x))
        x = self.fc_out(x)
        if self.normalise: # paper specifies no normalisation, but we keep the flag for experimentation
            x = F.normalize(x, dim=-1)
        return x


### InfoNCE + LogSumExp Regularisation Loss Function (Equation 3)

In [ ]:
def contrastive_loss(phi_sa, psi_sf_pos, psi_sf_neg, reg_coef=0.01):
    # Implements the contrastive RL objective from Eq. 3 of the paper:
    #   max  E[ log( e^{phi(s,a)^T psi(sf+)} / (e^{phi(s,a)^T psi(sf+)} + sum_j e^{phi(s,a)^T psi(sf_j-)}) )
    #           - 0.01 * log( sum_j e^{phi(s,a)^T psi(sf_j)} )^2 ]   <- LogSumExp regularisation
    #
    # phi_sa     : (B, d)  state-action embeddings phi(s, a)
    # psi_sf_pos : (B, d)  positive future-state embeddings psi(sf+), sampled via Geometric(1-γ) k steps ahead
    # psi_sf_neg : (B, d)  negative future-state embeddings psi(sf-), sampled uniformly from the replay buffer (marginal distribution)
    #
    # For each anchor phi_sa[i]:
    #   - one positive  : phi_sa[i]^T psi_sf_pos[i]   (col 0 of all_logits)
    #   - B negatives   : phi_sa[i]^T psi_sf_neg[j] for all j  (cols 1..B)
    # This keeps positives and negatives cleanly separated (no accidental positive-as-negative contamination).

    B = phi_sa.shape[0]

    pos_logits = (phi_sa * psi_sf_pos).sum(dim=-1, keepdim=True)  # (B, 1)  one dot-product per positive pair
    neg_logits = phi_sa @ psi_sf_neg.T                            # (B, B)  phi(s_i,a_i)^T psi(sf_j-) for all j

    all_logits = torch.cat([pos_logits, neg_logits], dim=1)  # (B, 1+B): column 0 is the positive
    labels = torch.zeros(B, dtype=torch.long, device=phi_sa.device)  # label 0 = column 0 = positive

    infonce_loss = F.cross_entropy(all_logits, labels)  # minimising this maximises the infoNCE term

    logsumexp = torch.logsumexp(all_logits, dim=-1)  # (B,), log sum_j exp(logit_j)
    reg_loss  = reg_coef * logsumexp.pow(2).mean()   # penalise large LogSumExp values (matches -0.01 * (...) in the maximisation objective)

    return infonce_loss + reg_loss  # minimise this to maximise the paper's contrastive objective


In [ ]:
MIN_STD = 1e-6
LOG_STD_MIN = float(np.log(MIN_STD))
LOG_STD_MAX =  2

class GoalConditionedActor(nn.Module): # goal-conditioned policy pi(a | s, g) — takes (s, g) and outputs a Gaussian action distribution

    def __init__(self, obs_dim, action_dim, hidden_dim=256, hidden_layers=5):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim * 2, hidden_dim)  # concatenate state s and goal g as input
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(hidden_layers - 2)])
        self.fc_mean    = nn.Linear(hidden_dim, action_dim)   # output the mean of the action distribution
        self.log_std_fc = nn.Linear(hidden_dim, action_dim)   # state-dependent log-std head

    def forward(self, obs, goal):
        x = torch.cat([obs, goal], dim=-1)  # (B, obs_dim * 2)
        x = F.relu(self.fc1(x))
        for layer in self.hidden_layers:
            x = F.relu(layer(x))
        mean    = self.fc_mean(x)  # (B, action_dim)
        log_std = self.log_std_fc(x).clamp(LOG_STD_MIN, LOG_STD_MAX)  # (B, action_dim)
        return mean, log_std

    def sample_action(self, obs, goal):
        mean, log_std = self.forward(obs, goal)
        std  = log_std.exp().clamp_min(MIN_STD)
        dist = torch.distributions.Normal(mean, std)

        # Reparameterised sample
        x_t    = dist.rsample()

        # Squash through tanh so actions always lie in (-1, 1)
        action = torch.tanh(x_t)

        # Correct log-prob for the tanh change-of-variables:
        # log π(a|s,g) = log N(x_t; μ, σ) - Σ log(1 - tanh²(x_t))
        log_prob = (dist.log_prob(x_t) - torch.log(1 - action.pow(2) + 1e-6)).sum(dim=-1, keepdim=True)  # (B, 1)
        entropy  = dist.entropy().sum(dim=-1, keepdim=True)  # (B, 1), H(pi(.|s,g))
        return action, log_prob, entropy



In [ ]:
# ── Imports for live visualisation ────────────────────────────────────────────
import matplotlib.pyplot as plt
from IPython.display import clear_output

# ── Hyperparameters ────────────────────────────────────────────────────────────
TOTAL_ENV_STEPS    = 30000
WARMUP_STEPS       = 2000
BATCH_SIZE         = 256
SAMPLES_PER_INSERT = 256
LR                 = 3e-4
INIT_ALPHA         = 0.1
ALPHA_LR           = 3e-4
TARGET_ENTROPY     = 0.0
MIN_STD            = 1e-6
RESIZE_FACTOR      = 4
ACTION_NOISE_STD   = 0.01
GOAL_TOLERANCE     = 0.75
LOG_INTERVAL       = 50
MAX_HORIZON        = 400
BUFFER_CAPACITY    = 10000
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

# ── Environment + single hard goal s* ─────────────────────────────────────────
env_train = ContinuousFourRoomsEnv(
    resize_factor=RESIZE_FACTOR,
    max_episode_steps=MAX_HORIZON,
    action_noise_std=ACTION_NOISE_STD,
    goal_tolerance=GOAL_TOLERANCE,
)

obs_dim    = env_train.observation_space.shape[0]   # state dimension only
action_dim = env_train.action_space.shape[0]
s_star     = torch.tensor(env_train.goal, device=DEVICE, dtype=torch.float32)

replay_buffer = TrajectoryReplayBuffer(
    capacity=BUFFER_CAPACITY,
    obs_dim=obs_dim,
    action_dim=action_dim,
    device=DEVICE,
)
print(DEVICE)
print(f"Training goal s*: {s_star.cpu().numpy()}")

phi_model = StateActionRepresentationModel(obs_dim, action_dim, hidden_dim=256, output_dim=64, hidden_layers=5, normalise=False).to(DEVICE)
psi_model = GoalRepresentationModel(obs_dim, hidden_dim=256, output_dim=64, hidden_layers=5, normalise=False).to(DEVICE)
actor     = GoalConditionedActor(obs_dim, action_dim, hidden_dim=256, hidden_layers=5).to(DEVICE)

critic_optimizer = optim.Adam(list(phi_model.parameters()) + list(psi_model.parameters()), lr=LR)
actor_optimizer  = optim.Adam(actor.parameters(), lr=LR)
log_alpha        = torch.tensor(np.log(INIT_ALPHA), dtype=torch.float32, device=DEVICE, requires_grad=True)
alpha_optimizer  = optim.Adam([log_alpha], lr=ALPHA_LR)

critic_loss_history     = []
actor_loss_history      = []
alpha_loss_history      = []
alpha_history           = []
ep_len_history          = []
entropy_history         = []
log_prob_history        = []
env_steps_at_update     = []
goal_similarity_history = []


def critic_update_step():
    batch = replay_buffer.sample_future_goal_batch(BATCH_SIZE)

    obs     = batch["obs"]
    actions = batch["actions"]
    goals   = batch["goals"]

    phi_sa    = phi_model(torch.cat([obs, actions], dim=-1))
    psi_g     = psi_model(goals)
    neg_goals = replay_buffer.sample_negative_future_goals(BATCH_SIZE)
    psi_neg   = psi_model(neg_goals)

    critic_loss = contrastive_loss(phi_sa, psi_g, psi_neg)

    critic_optimizer.zero_grad()
    critic_loss.backward()
    critic_optimizer.step()

    with torch.no_grad():
        psi_sstar = psi_model(s_star.unsqueeze(0).expand(BATCH_SIZE, -1))
        goal_sim  = (phi_sa.detach() * psi_sstar).sum(dim=-1).mean().item()

    return batch, critic_loss, goal_sim


def collect_episode(env, policy_fn):
    """Run one episode. policy_fn(obs_tensor) -> action_np. Returns (ep_dict, ep_steps)."""
    obs_t, _ = env.reset()
    done = False
    ep = {k: [] for k in ["obs", "actions", "rewards", "next_obs", "terminated", "truncated"]}
    while not done:
        action_np = policy_fn(obs_t)
        next_obs_t, reward, term, trunc, _ = env.step(action_np)
        ep["obs"].append(obs_t.astype(np.float32))
        ep["actions"].append(action_np.astype(np.float32))
        ep["rewards"].append(np.float32(reward))
        ep["next_obs"].append(next_obs_t.astype(np.float32))
        ep["terminated"].append(np.float32(term))
        ep["truncated"].append(np.float32(trunc))
        obs_t = next_obs_t
        done  = term or trunc
    return ep, len(ep["obs"])


total_env_steps = 0
print(f"Warmup: collecting {WARMUP_STEPS:,} random-action steps …")

def random_policy(obs_t):
    return env_train.action_space.sample()

while total_env_steps < WARMUP_STEPS:
    ep, ep_steps = collect_episode(env_train, random_policy)
    replay_buffer.add_episode(ep)
    total_env_steps += ep_steps

print(f"Warmup done. Buffer size: {replay_buffer.size:,} transitions ({total_env_steps:,} env steps collected).")

critic_pretrain_updates = max(1, int(np.ceil(SAMPLES_PER_INSERT * replay_buffer.size / BATCH_SIZE)))
print(f"Critic warmup: running {critic_pretrain_updates:,} critic-only replay updates before actor learning …")

for warmup_update in range(1, critic_pretrain_updates + 1):
    _, critic_loss, goal_sim = critic_update_step()
    if warmup_update % LOG_INTERVAL == 0 or warmup_update == critic_pretrain_updates:
        print(f"WarmupCritic {warmup_update:>5} / {critic_pretrain_updates:>5} | "
              f"critic_loss: {critic_loss.item():.4f} | goal_sim: {goal_sim:.4f}")

print("Critic warmup complete. Actor updates will start only from the online phase.")

grad_step = 0
print(f"Training: will run until {TOTAL_ENV_STEPS:,} total env steps …")

s_star_batch = s_star.unsqueeze(0)

while total_env_steps < TOTAL_ENV_STEPS:

    def actor_policy(obs_t):
        obs_tensor = torch.tensor(obs_t, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            action, _, _ = actor.sample_action(obs_tensor, s_star_batch)
        action_np = action.squeeze(0).cpu().numpy()
        return action_np

    ep, ep_steps = collect_episode(env_train, actor_policy)
    replay_buffer.add_episode(ep)
    total_env_steps += ep_steps

    updates_this_round = max(1, int(np.ceil(SAMPLES_PER_INSERT * ep_steps / BATCH_SIZE)))

    for _ in range(updates_this_round):
        grad_step += 1

        batch, critic_loss, goal_sim = critic_update_step()
        obs = batch["obs"]

        goal_similarity_history.append(goal_sim)

        sampled_actions, log_prob, entropy = actor.sample_action(obs, s_star_batch.expand(obs.shape[0], -1))

        phi_sa_actor = phi_model(torch.cat([obs, sampled_actions], dim=-1))
        psi_g_actor  = psi_model(s_star_batch.expand(obs.shape[0], -1))
        alpha        = log_alpha.exp()

        q_values   = (phi_sa_actor * psi_g_actor.detach()).sum(dim=-1, keepdim=True)
        actor_loss = (alpha.detach() * log_prob - q_values).mean()

        actor_optimizer.zero_grad()
        actor_loss.backward()
        actor_optimizer.step()

        alpha_loss = (alpha * (-log_prob.detach() - TARGET_ENTROPY)).mean()

        alpha_optimizer.zero_grad()
        alpha_loss.backward()
        alpha_optimizer.step()

        critic_loss_history.append(critic_loss.item())
        actor_loss_history.append(actor_loss.item())
        alpha_loss_history.append(alpha_loss.item())
        alpha_history.append(alpha.item())
        ep_len_history.append(ep_steps)
        entropy_history.append(entropy.detach().mean().item())
        log_prob_history.append(log_prob.detach().mean().item())
        env_steps_at_update.append(total_env_steps)

        if grad_step % LOG_INTERVAL == 0:
            print(f"GradStep {grad_step:>5} | env_steps: {total_env_steps:>7,} | ep_len: {ep_steps:>3} | "
                  f"updates/insert: {updates_this_round:>3} | critic_loss: {critic_loss.item():.4f} | "
                  f"actor_loss: {actor_loss.item():.4f} | alpha: {alpha.item():.6f} | goal_sim: {goal_sim:.4f}")

            clear_output(wait=True)
            fig, axes = plt.subplots(2, 2, figsize=(14, 8))
            axes = axes.flatten()

            axes[0].plot(env_steps_at_update, critic_loss_history, color='steelblue')
            axes[0].set_title("Critic loss (contrastive)")
            axes[0].set_xlabel("Env steps")
            axes[0].set_ylabel("Loss")

            axes[1].plot(env_steps_at_update, actor_loss_history, color='darkorange')
            axes[1].set_title("Actor loss")
            axes[1].set_xlabel("Env steps")
            axes[1].set_ylabel("Loss")

            axes[2].plot(env_steps_at_update, ep_len_history, color='seagreen')
            axes[2].axhline(MAX_HORIZON, color='gray', linestyle='--', linewidth=0.8, label='max horizon')
            axes[2].set_title("Episode length")
            axes[2].set_xlabel("Env steps")
            axes[2].set_ylabel("Steps")
            axes[2].legend()

            axes[3].plot(env_steps_at_update, goal_similarity_history, color='mediumpurple')
            axes[3].axhline(0, color='gray', linestyle='--', linewidth=0.8)
            axes[3].set_title(r"Goal similarity: $\phi(s,a)^	op\psi(s^*)$")
            axes[3].set_xlabel("Env steps")
            axes[3].set_ylabel("Mean dot-product")

            plt.suptitle(f"Four Rooms training progress — {total_env_steps:,} / {TOTAL_ENV_STEPS:,} env steps", fontsize=12)
            plt.tight_layout()
            plt.show()
            plt.close(fig)

env_train.close()
print("Training complete!")
print(f"Total env steps: {total_env_steps:,} | Online gradient updates: {grad_step:,}")
print(f"Replay buffer size: {replay_buffer.stats()['size']:,}")
print(f"Final alpha: {log_alpha.exp().item():.6f} | Target entropy: {TARGET_ENTROPY:.1f}")

### 5. Post-training summary plots for continuous Four Rooms

### 5a. Diagnostic: network I/O distributions and future-goal batches

In [ ]:
# ── Diagnostic: visualise network inputs, outputs, and one contrastive batch ──────
# Run this cell after training (phi_model, psi_model, replay_buffer must exist).

import matplotlib.pyplot as plt
import numpy as np
import torch

_DIAG_B = 512
phi_model.eval()
psi_model.eval()

with torch.no_grad():
    dbatch   = replay_buffer.sample_future_goal_batch(_DIAG_B)
    d_obs    = dbatch["obs"]
    d_act    = dbatch["actions"]
    d_goals  = dbatch["goals"]
    d_neg    = replay_buffer.sample_negative_future_goals(_DIAG_B)

    d_phi    = phi_model(torch.cat([d_obs, d_act], dim=-1))
    d_psi_p  = psi_model(d_goals)
    d_psi_n  = psi_model(d_neg)

    pos_logits = (d_phi * d_psi_p).sum(dim=-1).cpu().numpy()
    neg_logits = (d_phi @ d_psi_n.T).cpu().numpy().flatten()

    d_obs_np   = d_obs.cpu().numpy()
    d_act_np   = d_act.cpu().numpy()
    d_goals_np = d_goals.cpu().numpy()
    d_neg_np   = d_neg.cpu().numpy()
    d_phi_np   = d_phi.cpu().numpy()
    d_psi_p_np = d_psi_p.cpu().numpy()
    d_psi_n_np = d_psi_n.cpu().numpy()

    pos_goal_dist = np.linalg.norm(d_goals_np - s_star.cpu().numpy(), axis=1)
    neg_goal_dist = np.linalg.norm(d_neg_np - s_star.cpu().numpy(), axis=1)
    action_norms  = np.linalg.norm(d_act_np, axis=1)

fig, axes = plt.subplots(3, 3, figsize=(16, 12))

axes[0, 0].hist(d_obs_np[:, 0], bins=40, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0, 0].set_title('Input φ: state row')
axes[0, 0].set_xlabel(STATE_LABELS[0])

axes[0, 1].hist(d_obs_np[:, 1], bins=40, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0, 1].set_title('Input φ: state column')
axes[0, 1].set_xlabel(STATE_LABELS[1])

axes[0, 2].hist(action_norms, bins=40, color='coral', edgecolor='white', linewidth=0.3)
axes[0, 2].set_title('Input φ: action norm')
axes[0, 2].set_xlabel('||a||₂')

axes[1, 0].hist(d_phi_np[:, 0], bins=40, color='darkorange', edgecolor='white', linewidth=0.3, label='φ dim-0', alpha=0.7)
axes[1, 0].hist(d_phi_np[:, 1], bins=40, color='gold', edgecolor='white', linewidth=0.3, label='φ dim-1', alpha=0.7)
axes[1, 0].set_title('Output φ(s,a): first two dimensions')
axes[1, 0].legend()

axes[1, 1].hist(d_psi_p_np[:, 0], bins=40, color='seagreen', edgecolor='white', linewidth=0.3, label='ψ(sf+) dim-0', alpha=0.7)
axes[1, 1].hist(d_psi_p_np[:, 1], bins=40, color='limegreen', edgecolor='white', linewidth=0.3, label='ψ(sf+) dim-1', alpha=0.7)
axes[1, 1].set_title('Output ψ(positive future goal): first two dimensions')
axes[1, 1].legend()

axes[1, 2].hist(d_psi_n_np[:, 0], bins=40, color='mediumpurple', edgecolor='white', linewidth=0.3, label='ψ(sf-) dim-0', alpha=0.7)
axes[1, 2].hist(d_psi_n_np[:, 1], bins=40, color='plum', edgecolor='white', linewidth=0.3, label='ψ(sf-) dim-1', alpha=0.7)
axes[1, 2].set_title('Output ψ(negative future goal): first two dimensions')
axes[1, 2].legend()

axes[2, 0].hist(pos_logits, bins=40, color='seagreen', edgecolor='white', linewidth=0.3, label=f'positive (n={len(pos_logits)})', alpha=0.8)
axes[2, 0].hist(neg_logits, bins=80, color='mediumpurple', edgecolor='white', linewidth=0.2, label=f'negative (n={len(neg_logits)})', alpha=0.5)
axes[2, 0].axvline(np.mean(pos_logits), color='darkgreen', linestyle='--', linewidth=1.5, label=f'μ_pos={np.mean(pos_logits):.2f}')
axes[2, 0].axvline(np.mean(neg_logits), color='darkviolet', linestyle='--', linewidth=1.5, label=f'μ_neg={np.mean(neg_logits):.2f}')
axes[2, 0].set_title('Logit distributions: positive vs negative')
axes[2, 0].set_xlabel('φ(s,a) · ψ(g)')
axes[2, 0].legend(fontsize=8)

axes[2, 1].scatter(d_phi_np[:, 0], d_phi_np[:, 1], c='darkorange', s=8, alpha=0.5, label='φ(s,a)')
axes[2, 1].scatter(d_psi_p_np[:, 0], d_psi_p_np[:, 1], c='seagreen', s=8, alpha=0.5, label='ψ(sf+)')
axes[2, 1].scatter(d_psi_n_np[:, 0], d_psi_n_np[:, 1], c='mediumpurple', s=8, alpha=0.3, label='ψ(sf-)')
with torch.no_grad():
    psi_star_np = psi_model(s_star.unsqueeze(0)).cpu().numpy()
axes[2, 1].scatter(psi_star_np[0, 0], psi_star_np[0, 1], c='red', s=120, marker='*', zorder=5, label='ψ(s*) goal')
axes[2, 1].set_title('Representation space (first two dimensions)')
axes[2, 1].set_xlabel('dim 0')
axes[2, 1].set_ylabel('dim 1')
axes[2, 1].legend(fontsize=8)

axes[2, 2].hist(pos_goal_dist, bins=40, color='seagreen', edgecolor='white', linewidth=0.3, label='positive future goals', alpha=0.7)
axes[2, 2].hist(neg_goal_dist, bins=40, color='mediumpurple', edgecolor='white', linewidth=0.3, label='negative future goals', alpha=0.7)
axes[2, 2].set_title('Distance to s*: positive vs negative goals')
axes[2, 2].set_xlabel('||g - s*||₂')
axes[2, 2].legend(fontsize=8)

plt.suptitle(
    f'Network I/O diagnostic | buffer={replay_buffer.size:,} transitions | state_dim={obs_dim} | normalise={phi_model.normalise}',
    fontsize=11
)
plt.tight_layout()
plt.savefig('/tmp/network_io_diagnostic_gridworld.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

phi_model.train()
psi_model.train()

print(f'
Summary statistics:')
print(f'  φ(s,a) output  — mean: {d_phi_np.mean():.3f}  std: {d_phi_np.std():.3f}  min: {d_phi_np.min():.3f}  max: {d_phi_np.max():.3f}')
print(f'  ψ(sf+) output  — mean: {d_psi_p_np.mean():.3f}  std: {d_psi_p_np.std():.3f}  min: {d_psi_p_np.min():.3f}  max: {d_psi_p_np.max():.3f}')
print(f'  ψ(sf-) output  — mean: {d_psi_n_np.mean():.3f}  std: {d_psi_n_np.std():.3f}  min: {d_psi_n_np.min():.3f}  max: {d_psi_n_np.max():.3f}')
print(f'  pos logits     — mean: {pos_logits.mean():.3f}  std: {pos_logits.std():.3f}')
print(f'  neg logits     — mean: {neg_logits.mean():.3f}  std: {neg_logits.std():.3f}')
print(f'  contrastive gap (μ_pos - μ_neg): {pos_logits.mean() - neg_logits.mean():.3f}')

In [ ]:
# ── Post-training summary: four-panel static plot ─────────────────────────────
# Smooth a 1-D list with a simple rolling window mean to reduce noise in the curves.
def smooth(values, window=10):
    if len(values) < window:
        return values
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode='valid')

steps_full = np.asarray(env_steps_at_update)
smooth_w   = max(1, len(steps_full) // 20) if len(steps_full) > 0 else 1

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# ─ Critic loss ─
ax = axes[0, 0]
ax.plot(steps_full, critic_loss_history, color='steelblue', alpha=0.3, linewidth=0.8, label='raw')
ax.plot(steps_full[smooth_w - 1:], smooth(critic_loss_history, smooth_w),
        color='steelblue', linewidth=2, label=f'smoothed (w={smooth_w})')
ax.set_title("Critic loss (contrastive, Eq. 3)")
ax.set_xlabel("Env steps")
ax.set_ylabel("Loss")
ax.legend()

# ─ Actor loss ─
ax = axes[0, 1]
ax.plot(steps_full, actor_loss_history, color='darkorange', alpha=0.3, linewidth=0.8, label='raw')
ax.plot(steps_full[smooth_w - 1:], smooth(actor_loss_history, smooth_w),
        color='darkorange', linewidth=2, label=f'smoothed (w={smooth_w})')
ax.set_title("Actor loss (Eq. 4)")
ax.set_xlabel("Env steps")
ax.set_ylabel("Loss")
ax.legend()

# ─ Episode length ─
ax = axes[1, 0]
ax.plot(steps_full, ep_len_history, color='seagreen', alpha=0.3, linewidth=0.8, label='raw')
ax.plot(steps_full[smooth_w - 1:], smooth(ep_len_history, smooth_w),
        color='seagreen', linewidth=2, label=f'smoothed (w={smooth_w})')
ax.axhline(MAX_HORIZON, color='gray', linestyle='--', linewidth=0.8, label=f'max horizon ({MAX_HORIZON})')
ax.set_title("Episode length")
ax.set_xlabel("Env steps")
ax.set_ylabel("Steps per episode")
ax.legend()
# A shorter episode that terminates (rather than truncates) means the agent reached the goal.

# ─ Policy entropy ─
ax = axes[1, 1]
ax.plot(steps_full, entropy_history, color='mediumpurple', alpha=0.3, linewidth=0.8, label='raw')
ax.plot(steps_full[smooth_w - 1:], smooth(entropy_history, smooth_w),
        color='mediumpurple', linewidth=2, label=f'smoothed (w={smooth_w})')
ax.axhline(TARGET_ENTROPY, color='gray', linestyle='--', linewidth=0.8, label=f'target entropy ({TARGET_ENTROPY:.1f})')
ax.set_title("Policy entropy  H(π(·|s, g))")
ax.set_xlabel("Env steps")
ax.set_ylabel("Entropy (nats)")
ax.legend()
# With target entropy = 0, the policy entropy should trend downward toward zero over training.

plt.suptitle("Post-training summary", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()



### 6. State-coverage heatmap in Four Rooms

In [ ]:
# ── State-coverage heatmap ─────────────────────────────────────────────────────
# Visualise where the agent has visited inside the continuous Four Rooms map.

all_states = replay_buffer.obs[: replay_buffer.size]
rows = all_states[:, 0]
cols = all_states[:, 1]

fig, ax = plt.subplots(figsize=(8, 8))

h = ax.hist2d(
    cols,
    rows,
    bins=[env_train.width, env_train.height],
    range=[[0, env_train.width], [0, env_train.height]],
    cmap='YlOrRd',
    density=False,
)
ax.imshow(
    env_train.walls,
    origin='lower',
    extent=[0, env_train.width, 0, env_train.height],
    cmap='binary',
    alpha=0.25,
    interpolation='nearest',
)

cbar = plt.colorbar(h[3], ax=ax)
cbar.set_label('Visit count')

ax.scatter(s_star[1].item(), s_star[0].item(), c='lime', s=120, marker='*', edgecolors='black', linewidths=0.5, label='goal s*')
ax.set_xlabel(STATE_LABELS[1])
ax.set_ylabel(STATE_LABELS[0])
ax.set_title('State-coverage heatmap (all replay-buffer transitions)')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(0, env_train.width)
ax.set_ylim(0, env_train.height)

plt.tight_layout()
plt.show()
print(f'Plotted {replay_buffer.size:,} transitions.')

### 7. Final policy rollout — plot learned trajectories in Four Rooms

In [ ]:
# ── Roll out the learned policy and plot trajectories ─────────────────────────
# This cell visualises policy rollouts directly in the Four Rooms map.

EVAL_EPISODES = 5

env_eval = ContinuousFourRoomsEnv(
    resize_factor=RESIZE_FACTOR,
    max_episode_steps=MAX_HORIZON,
    action_noise_std=ACTION_NOISE_STD,
    goal_tolerance=GOAL_TOLERANCE,
)

phi_model.eval()
psi_model.eval()
actor.eval()

ep_rewards = []
ep_successes = []
ep_lengths = []
ep_rollouts = []
ep_distances = []
ep_action_norms = []

s_star_render = s_star.unsqueeze(0)

for ep_i in range(EVAL_EPISODES):
    obs_r, info_r = env_eval.reset()
    done_r = False
    ep_rew = 0.0
    states_this_ep = [obs_r.copy()]
    distances_this_ep = [float(np.linalg.norm(obs_r - env_eval.goal))]
    action_norms_this_ep = []

    while not done_r:
        obs_tensor_r = torch.tensor(obs_r, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            action_r, _, _ = actor.sample_action(obs_tensor_r, s_star_render)
        action_r_np = action_r.squeeze(0).cpu().numpy()
        action_r_np = np.clip(action_r_np, env_eval.action_space.low, env_eval.action_space.high)

        obs_r, rew_r, term_r, trunc_r, info_r = env_eval.step(action_r_np)
        ep_rew += rew_r
        states_this_ep.append(obs_r.copy())
        distances_this_ep.append(float(info_r['distance_to_goal']))
        action_norms_this_ep.append(float(np.linalg.norm(action_r_np)))
        done_r = term_r or trunc_r

    rollout = np.asarray(states_this_ep, dtype=np.float32)
    ep_rollouts.append(rollout)
    ep_distances.append(np.asarray(distances_this_ep, dtype=np.float32))
    ep_action_norms.append(np.asarray(action_norms_this_ep, dtype=np.float32))
    ep_rewards.append(ep_rew)
    ep_successes.append(bool(ep_rew > 0))
    ep_lengths.append(len(states_this_ep) - 1)

    print(f"Rollout {ep_i + 1}: {ep_lengths[-1]} steps | total reward: {ep_rew:.1f} | success: {ep_successes[-1]}")

env_eval.close()
phi_model.train()
psi_model.train()
actor.train()

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(
    env_train.walls,
    origin='lower',
    extent=[0, env_train.width, 0, env_train.height],
    cmap='binary',
    alpha=0.30,
    interpolation='nearest',
)

colours = plt.cm.tab10.colors
for i, rollout in enumerate(ep_rollouts):
    ax.plot(rollout[:, 1], rollout[:, 0], color=colours[i % len(colours)], linewidth=1.6, label=f'Rollout {i + 1}')
    ax.scatter(rollout[0, 1], rollout[0, 0], color=colours[i % len(colours)], s=24, marker='o')

ax.scatter(s_star[1].item(), s_star[0].item(), c='lime', s=140, marker='*', edgecolors='black', linewidths=0.5, label='goal s*')
ax.set_xlabel(STATE_LABELS[1])
ax.set_ylabel(STATE_LABELS[0])
ax.set_title('Learned policy rollouts in continuous Four Rooms')
ax.set_xlim(0, env_train.width)
ax.set_ylim(0, env_train.height)
ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig('/tmp/four_rooms_rollouts.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Success rate: {np.mean(ep_successes):.2%} | mean episode length: {np.mean(ep_lengths):.1f}")
print("Saved rollout plot to /tmp/four_rooms_rollouts.png")

### 8. Per-rollout distance-to-goal and action-size plots

In [ ]:
# ── Distance-to-goal and action norm over time for each rollout ───────────────

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=False)

colours = plt.cm.tab10.colors

for i, dist_traj in enumerate(ep_distances):
    ax1.plot(dist_traj, color=colours[i % len(colours)], linewidth=1.5, label=f'Rollout {i + 1} (success={ep_successes[i]})')

ax1.axhline(GOAL_TOLERANCE, color='lime', linestyle='--', linewidth=1.2, label=f'goal tolerance ({GOAL_TOLERANCE})')
ax1.set_ylabel('Distance to s*')
ax1.set_title('Learned policy — distance to the goal over time')
ax1.legend(fontsize=9)

for i, action_norm_traj in enumerate(ep_action_norms):
    ax2.plot(action_norm_traj, color=colours[i % len(colours)], linewidth=1.5, label=f'Rollout {i + 1}')

ax2.axhline(1.0, color='gray', linestyle=':', linewidth=0.8, label='max per-axis magnitude = 1')
ax2.set_xlabel('Timestep')
ax2.set_ylabel('Action norm')
ax2.set_title('Learned policy — action magnitude over time')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()